# Dataset (II) EDA
## Context
Dataset (I) (containing bus actions such as 'arriving' or 'departing' and bus stops) has several stops missing from every trip. Therefore, calculating travel time between every sequential stop impossible. 
## Goal
- Explore dataset (II) (containing bus GPS location data)
- Map each GPS location (longitude, latitude) to the nearest bus stop to see if it solves the missing bus stops problem
# Import packages

In [2]:
# to import from local modules
from pathlib import Path
import sys  

parent_dir = str(Path().resolve().parents[0])
sys.path.insert(0, parent_dir)

In [ ]:
import polars as pl
from src.constants import ROUTES, STOPS, SAMPLE, DATA_FILE
import numpy as np
from scipy.spatial import KDTree

pl.Config.set_tbl_rows(30)

polars.config.Config

# Load datasets

In [4]:
df_routes = pl.read_csv(ROUTES)
df_stops = pl.read_csv(STOPS)
df_raw = pl.scan_csv(
    SAMPLE, ignore_errors=True, schema_overrides={"SubRouteID": pl.String}
)

In [4]:
df_stops.glimpse()

Rows: 19439
Columns: 19
$ StopUID          <str> 'THB267193', 'THB267018', 'THB307459', 'THB300514', 'THB168103', 'THB129966', 'THB129964', 'THB271131', 'THB214833', 'THB290370'
$ StopID           <i64> 267193, 267018, 307459, 300514, 168103, 129966, 129964, 271131, 214833, 290370
$ AuthorityID      <i64> 10, 10, 10, 10, 10, 10, 10, 10, 10, 10
$ StopNameZh_tw    <str> '仁愛新生路口', '光華商場', '光華商場', '臺北車站(鄭州)', '酒泉重慶路口', '酒泉重慶路口', '昌吉重慶路口', '昌吉重慶路口', '捷運大橋頭站', '圓環(南京)'
$ StopNameEn       <str> 'Ren-ai and Xinsheng Intersection', 'Guanghua Computer Market', 'Guanghua Market', 'Taipei Station (Zhengzhou)', 'Jiuquan Chongqing Road Intersection', 'Jiuquan Chongqing Road Intersection', 'Changji Chongqing Road Intersection', 'Changji Chongqing Road Intersection', 'MRT Daqiaotou Station', 'Yuanhuan(Nanking)'
$ PositionLat      <f64> 25.03789, 25.044621, 25.044532, 25.048361, 25.07181, 25.07173, 25.06567, 25.06556, 25.06344, 25.053656
$ PositionLon      <f64> 121.53288, 121.533089, 121.532734, 121.5

# High-Performance Spatial Mapping: GPS Pings to Bus Stops
Mapping millions of GPS pings to ~19,000 bus stops is a $O(N \times M)$ problem that is computationally prohibitive via brute force. 

**Technical Approach:**
* **Algorithmic Efficiency:** Implemented a **KDTree** to reduce search complexity to **$O(N \log M)$**.
* **Geometric Precision:** Converted Geodetic (Lat/Lon) coordinates to **3D Cartesian (ECEF)** spaceto accurately calculate the true distance.
* **Vectorized Pipeline:** Utilized **Polars `map_batches`** and **NumPy** to process data in high-speed chunks.
* `pl.LazeFrame`: Use lazy mode to prepare for future scaling to full dataset

In [5]:
def assign_nearest_stop(
    df_bus: pl.LazyFrame,
    df_stops: pl.DataFrame,
) -> pl.LazyFrame:
    """
    For every row in `df_bus`, find the nearest stop in `df_stops` using a
    KDTree built on Cartesian (ECEF) coordinates, and add:
        - `out_name_col`  : StopNameZh_tw of the nearest stop
        - `out_id_col`    : StopUID of the nearest stop
        - `out_dist_col`  : great-circle distance in metres (float, 2 d.p.)

    Converting lat/lon → 3D Cartesian (ECEF) before building the KDTree to compare the distance more accurately

    Parameters
    ----------
    df_bus   : GPS ping DataFrame
    df_stops : Bus stop reference DataFrame
    """
    R = 6_371_000.0  # Earth radius in metres

    def to_cartesian(lat_deg: np.ndarray, lon_deg: np.ndarray) -> np.ndarray:
        """Convert lat/lon (degrees) to ECEF Cartesian coordinates."""
        lat = np.radians(lat_deg)
        lon = np.radians(lon_deg)
        x = R * np.cos(lat) * np.cos(lon)
        y = R * np.cos(lat) * np.sin(lon)
        z = R * np.sin(lat)
        return np.column_stack((x, y, z))

    # --- build KDTree from stop Cartesian coords (done once) ---
    stop_lats = df_stops["PositionLat"].to_numpy()
    stop_lons = df_stops["PositionLon"].to_numpy()
    stop_names = df_stops["StopNameZh_tw"].to_numpy()
    stop_ids = df_stops["StopUID"].to_numpy()

    stop_cart = to_cartesian(stop_lats, stop_lons)
    tree = KDTree(stop_cart)

    def find_nearest_in_batch(struct_series: pl.Series) -> pl.Series:
        # struct_series contains both Lat and Lon fields
        lats = struct_series.struct.field("PositionLat").to_numpy()
        lons = struct_series.struct.field("PositionLon").to_numpy()

        # Convert this batch to Cartesian
        gps_cart = to_cartesian(lats, lons)

        # Query the pre-built tree (using all cores)
        euclidean_dists, indices = tree.query(gps_cart, workers=1)

        # Calculate Arc Distance
        arc_dists = 2 * R * np.arcsin(np.clip(euclidean_dists / (2 * R), -1.0, 1.0))

        # Return a Struct so we can unnest multiple columns at once
        return pl.DataFrame(
            {
                "NearestStopName": stop_names[indices].astype(str),
                "NearestStopUID": stop_ids[indices].astype(str),
                "DistanceToStop_meter": np.round(arc_dists, 2),
            }
        ).to_struct("nearest_results")

    return df_bus.with_columns(
        pl.struct(["PositionLat", "PositionLon"])
        .map_batches(
            find_nearest_in_batch,
            return_dtype=pl.Struct(
                {
                    "NearestStopName": pl.String,
                    "NearestStopUID": pl.String,
                    "DistanceToStop_meter": pl.Float64,
                }
            ),
            is_elementwise=False,
        )
        .alias("result_struct")
    ).unnest("result_struct")

In [6]:
result = assign_nearest_stop(df_raw, df_stops).collect(engine="streaming")

Entire assighment on 700k+ rows took 1.2 seconds. (extremely scalable)

## Singe Route Inspection
Use Google Map to check if the bus stop assignments and distances calculated are correct.

In [7]:
result.filter(
    pl.col("RouteID") == 1728,
    pl.col("Direction") == "返程",
    pl.col("DutyStatus") == "開始",
    pl.col("PlateNumb") == "058-FS",
    pl.col("GPSTime").str.starts_with("2026-03-01T06"),
    pl.col("DistanceToStop_meter") < 100,
).select(
    [
        "PlateNumb",
        "RouteID",
        "Direction",
        "PositionLon",
        "PositionLat",
        "GPSTime",
        "NearestStopName",
        "DistanceToStop_meter",
    ]
).sort("GPSTime")

PlateNumb,RouteID,Direction,PositionLon,PositionLat,GPSTime,NearestStopName,DistanceToStop_meter
str,i64,str,f64,f64,str,str,f64
"""058-FS""",1728,"""返程""",120.972773,24.801118,"""2026-03-01T06:00:31+08:00""","""新竹轉運站""",29.3
"""058-FS""",1728,"""返程""",120.973008,24.801027,"""2026-03-01T06:00:51+08:00""","""新竹轉運站""",47.04
"""058-FS""",1728,"""返程""",120.974152,24.80215,"""2026-03-01T06:01:31+08:00""","""新竹轉運站""",62.13
"""058-FS""",1728,"""返程""",120.97447,24.801753,"""2026-03-01T06:02:11+08:00""","""新竹轉運站""",26.13
"""058-FS""",1728,"""返程""",120.979095,24.802617,"""2026-03-01T06:03:31+08:00""","""公園站""",57.74
"""058-FS""",1728,"""返程""",120.985165,24.801593,"""2026-03-01T06:06:31+08:00""","""二分局""",32.26
"""058-FS""",1728,"""返程""",120.986803,24.801068,"""2026-03-01T06:06:51+08:00""","""二分局""",82.4
"""058-FS""",1728,"""返程""",120.989342,24.800242,"""2026-03-01T06:08:33+08:00""","""工研院光復院區""",26.72
"""058-FS""",1728,"""返程""",120.990622,24.79983,"""2026-03-01T06:09:13+08:00""","""馬偕醫院""",22.49


## Finding
- The bus stop assignments and distances are accurate.
- Bus stops that aren't on this route are assigned here.
- Solution. When assigning bus stops, only query from stops that belong to that route.

In [8]:
def assign_nearest_stop(
    df_bus: pl.LazyFrame,
    df_routes: pl.DataFrame,
) -> pl.LazyFrame:
    """
    For every row in `df_bus`, find the nearest stop in `df_stops` using a
    KDTree built on Cartesian (ECEF) coordinates

    Converting lat/lon → 3D Cartesian (ECEF) before building the KDTree to compare the distance more accurately

    Parameters
    ----------
    df_bus   : GPS ping DataFrame
    df_stops : Bus stop reference DataFrame
    """
    R = 6_371_000.0

    def to_cartesian(lat_deg: np.ndarray, lon_deg: np.ndarray) -> np.ndarray:
        lat = np.radians(lat_deg)
        lon = np.radians(lon_deg)
        x = R * np.cos(lat) * np.cos(lon)
        y = R * np.cos(lat) * np.sin(lon)
        z = R * np.sin(lat)
        return np.column_stack((x, y, z))

    # --- build one KDTree per SubRouteID ---
    route_trees: dict[str, KDTree] = {}
    route_stop_names: dict[str, np.ndarray] = {}
    route_stop_ids: dict[str, np.ndarray] = {}

    df_routes_unique = df_routes.unique(subset=["SubRouteID", "StopNameZh_tw"]).select(
        ["SubRouteID", "PositionLat", "PositionLon", "StopNameZh_tw", "StopUID"]
    )

    for route_id, group in df_routes_unique.group_by("SubRouteID"):
        rid = str(route_id[0])
        stop_lats = group["PositionLat"].to_numpy()
        stop_lons = group["PositionLon"].to_numpy()
        route_trees[rid] = KDTree(to_cartesian(stop_lats, stop_lons))
        route_stop_names[rid] = group["StopNameZh_tw"].to_numpy().astype(str)
        route_stop_ids[rid] = group["StopUID"].to_numpy().astype(str)

    def find_nearest_in_batch(struct_series: pl.Series) -> pl.Series:
        lats = struct_series.struct.field("PositionLat").to_numpy()
        lons = struct_series.struct.field("PositionLon").to_numpy()
        route_ids = struct_series.struct.field("SubRouteID").to_numpy().astype(str)

        gps_cart = to_cartesian(lats, lons)

        result_names = np.empty(len(lats), dtype=object)
        result_ids = np.empty(len(lats), dtype=object)
        result_dists = np.empty(len(lats), dtype=np.float64)

        # Group indices by SubRouteID to batch-query each tree once
        unique_rids, inverse = np.unique(route_ids, return_inverse=True)
        for i, rid in enumerate(unique_rids):
            mask = inverse == i
            if rid not in route_trees:
                result_names[mask] = ""
                result_ids[mask] = ""
                result_dists[mask] = np.nan
                continue

            euclidean_dists, indices = route_trees[rid].query(gps_cart[mask], workers=1)
            # calculate the arc length (in meters) between two stops
            arc_dists = 2 * R * np.arcsin(np.clip(euclidean_dists / (2 * R), -1.0, 1.0))

            result_names[mask] = route_stop_names[rid][indices]
            result_ids[mask] = route_stop_ids[rid][indices]
            result_dists[mask] = np.round(arc_dists, 2)

        return pl.DataFrame(
            {
                "NearestStopName": result_names.astype(str),
                "NearestStopUID": result_ids.astype(str),
                "DistanceToStop_meter": result_dists,
            }
        ).to_struct("nearest_results")

    return df_bus.with_columns(
        pl.struct(["PositionLat", "PositionLon", "SubRouteID"])
        .map_batches(
            find_nearest_in_batch,
            return_dtype=pl.Struct(
                {
                    "NearestStopName": pl.String,
                    "NearestStopUID": pl.String,
                    "DistanceToStop_meter": pl.Float64,
                }
            ),
        )
        .alias("result_struct")
    ).unnest("result_struct")

In [9]:
result = assign_nearest_stop(df_raw, df_routes).collect()
result.head()

PlateNumb,OperatorID,OperatorNo,RouteUID,RouteID,RouteNameZh_tw,RouteNameEn,SubRouteUID,SubRouteID,SubRouteNameZh_tw,SubRouteNameEn,Direction,PositionLon,PositionLat,GeoHash,Speed,Azimuth,DutyStatus,BusStatus,MessageType,GPSTime,TransTime,SrcRecTime,SrcTransTime,SrcUpdateTime,UpdateTime,NearestStopName,NearestStopUID,DistanceToStop_meter
str,i64,i64,str,i64,i64,i64,str,str,i64,i64,str,f64,f64,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,f64
"""393-FL""",35,1308,"""THB0968""",968,968,968,"""THB096802""","""096802""",968,968,"""返程""",121.288387,25.041882,"""wsqnryxx8""",4,298,"""開始""","""正常""","""定期""","""2026-03-01T19:07:27+08:00""",null,"""2026-03-01T19:07:27+08:00""","""2026-03-01T19:07:27+08:00""",null,"""2026-03-01T19:07:29+08:00""","""南昌路口""","""THB300193""",106.76
"""393-FL""",35,1308,"""THB0968""",968,968,968,"""THB096802""","""096802""",968,968,"""返程""",121.51335,25.068467,"""wsqqt5e66""",23,335,"""開始""","""正常""","""定期""","""2026-03-01T09:03:22+08:00""",null,"""2026-03-01T09:03:23+08:00""","""2026-03-01T09:03:23+08:00""",null,"""2026-03-01T09:03:24+08:00""","""庫倫街口""","""THB300198""",571.0
"""805-FZ""",35,1308,"""THB0968""",968,968,968,"""THB096802""","""096802""",968,968,"""返程""",121.276112,25.033428,"""wsqnrtnw6""",48,254,"""正常""","""正常""","""定期""","""2026-03-01T09:03:24+08:00""",null,"""2026-03-01T09:03:24+08:00""","""2026-03-01T09:03:24+08:00""",null,"""2026-03-01T09:03:29+08:00""","""南竹蘆興街口""","""THB300191""",535.92
"""393-FL""",35,1308,"""THB0968""",968,968,968,"""THB096801""","""096801""",968,968,"""去程""",121.257787,25.016297,"""wsqnr614r""",0,0,"""結束""","""未知""","""定期""","""2026-03-01T13:53:15+08:00""",null,"""2026-03-01T13:53:15+08:00""","""2026-03-01T13:53:15+08:00""",null,"""2026-03-01T13:53:19+08:00""","""大竹消防隊""","""THB300177""",427.35
"""393-FL""",35,1308,"""THB0968""",968,968,968,"""THB096802""","""096802""",968,968,"""返程""",121.292528,25.043478,"""wsqq2p48b""",0,314,"""結束""","""未知""","""定期""","""2026-03-01T15:58:07+08:00""",null,"""2026-03-01T15:58:07+08:00""","""2026-03-01T15:58:07+08:00""",null,"""2026-03-01T15:58:09+08:00""","""長榮""","""THB300197""",311.04


In [10]:
# check the overall assignment
result.select(
    ["NearestStopName", "NearestStopUID", "DistanceToStop_meter"]
).null_count()

NearestStopName,NearestStopUID,DistanceToStop_meter
u32,u32,u32
0,0,0


## Finding
- Now the bus stops are correctly assigned to the route they are on.
- No null values indicates successful bulk assignment.
# Did it solve missing stop problem?

In [11]:
pl.Config.set_tbl_rows(141) # to see the entirety of one trip

result.filter(
    pl.col("SubRouteID") == "172802",
    pl.col("Direction") == "返程",
    pl.col("DutyStatus") == "開始",
    pl.col("PlateNumb") == "058-FS",
).sort("GPSTime").select(
    [
        "PlateNumb",
        "SubRouteID",
        "Direction",
        "GPSTime",
        "NearestStopName",
        "DistanceToStop_meter",
    ]
).filter(
    pl.col("GPSTime").str.starts_with("2026-03-01T06")
    | pl.col("GPSTime").str.starts_with("2026-03-01T07")
)


PlateNumb,SubRouteID,Direction,GPSTime,NearestStopName,DistanceToStop_meter
str,str,str,str,str,f64
"""058-FS""","""172802""","""返程""","""2026-03-01T06:00:31+08:00""","""新竹轉運站""",31.52
"""058-FS""","""172802""","""返程""","""2026-03-01T06:00:51+08:00""","""新竹轉運站""",48.48
"""058-FS""","""172802""","""返程""","""2026-03-01T06:01:31+08:00""","""新竹轉運站""",211.82
"""058-FS""","""172802""","""返程""","""2026-03-01T06:02:11+08:00""","""新竹轉運站""",215.25
"""058-FS""","""172802""","""返程""","""2026-03-01T06:02:51+08:00""","""公園站""",291.61
"""058-FS""","""172802""","""返程""","""2026-03-01T06:03:31+08:00""","""公園站""",58.2
"""058-FS""","""172802""","""返程""","""2026-03-01T06:03:51+08:00""","""公園站""",145.25
"""058-FS""","""172802""","""返程""","""2026-03-01T06:04:31+08:00""","""公園站""",147.06
"""058-FS""","""172802""","""返程""","""2026-03-01T06:05:11+08:00""","""公園站""",149.64


## Conclusion
- Despite more complete data per bus trip, it's still impossible to accurately pinpoint when a bus actually goes to/by *every* stop. 
- Possible reasons include:
    - GPS measurement error
    - Two bus stops are too close to each other
- The situation will likely worsen for longer routes or routes that visit rural areas.
- This means certain missing stops might inevitably have to be ignored and only more popular stops will be considered.
- **Decision**: The data quality is not better for this dataset and using it would introduce more complexity (and potential errors). Therefore, for the main pipeline, Dataset (I) will be used. 
